In [15]:
import os
import sqlite3
from dotenv import load_dotenv

from langchain.prompts import ChatPromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain.schema import HumanMessage, AIMessage, SystemMessage
from langchain.chains import ConversationChain
from langchain_groq import ChatGroq

# Load environment variables
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# LLM initialization
llm = ChatGroq(model="deepseek-r1-distill-llama-70b", temperature=0)

# Memory for maintaining conversation context
memory = ConversationBufferMemory(return_messages=True)

# Prompt to summarize SQL results with self-improvement advice
sql_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an assistant that summarizes SQL query results to help the user improve their habits. Focus on identifying habits that need attention and explain how the user can improve their consistency."
    "SQL input is in the form: HabitID, Description, Priority, Preferences,Type,Time, Remarks"),
    ("human", "{input}")
])

# General Personal Assistant prompt
general_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a Personal Assistant that helps the user improve on himself, by giving him advice on how to be more productive, more organized, and more efficient. 
You are friendly, patient, and insightful. You understand the user's context and provide actionable, thoughtful suggestions.

Start by analyzing their current habits based on tracked data and guide them step by step toward improvement.
"""),
    ("human", "{input}")
])

# SQL Retriever Function
def retrieve_low_completion_habits():
    conn = sqlite3.connect('student.db')
    cursor = conn.cursor()

    query = """
    SELECT H.* FROM Habits H
    JOIN HabitTimes HT ON H.ID = HT.HabitID
    WHERE HT.No_of_days_Completed < HT.Total_no_of_days * 3 / 4;
    """
    cursor.execute(query)
    results = cursor.fetchall()
    for row in results:
        print(row)
    conn.close()
    return results

# SQL Summary Chain
def get_sql_summary_with_context(sql_results):
    # Create prompt with memory context
    input_text = f"Summarize these habit records and give suggestions: {sql_results}"

    chain = sql_prompt | llm
    summary = chain.invoke({"input": input_text})

    # Save conversation for continuity
    memory.save_context({"input": input_text}, {"output": summary.content})

    return summary.content

# General Personal Assistant Chain
def get_general_advice(user_input):
    chain = general_prompt | llm
    response = chain.invoke({"input": user_input})

    memory.save_context({"input": user_input}, {"output": response.content})
    return response.content

# Example Workflow
if __name__ == "__main__":
    # Retrieve habit data needing attention
    habit_data = retrieve_low_completion_habits()

    # Get summary and advice based on SQL results
    sql_summary = get_sql_summary_with_context(habit_data)
    print("🔍 Habit Analysis:\n", sql_summary)

    # Optionally: get further personalized advice from user
    user_input = "I want to improve my routine. What should I focus on first?"
    assistant_response = get_general_advice(user_input)
    print("\n🧠 Assistant Advice:\n", assistant_response)



(1, 'Morning Exercise', 1, 5, 'Health', '06:00', 'An urgent new task has come up.')
(2, 'Read Books', 2, 4, 'Learning', '20:00', None)
(3, 'Practice Piano', 3, 3, 'Creativity', '15:00', None)
🔍 Habit Analysis:
 <think>
Okay, so the user has provided some SQL query results about their habits. I need to summarize these and give suggestions to help them improve their consistency. Let me start by understanding the data.

First, looking at the habits: Morning Exercise, Read Books, and Practice Piano. Each has a HabitID, Description, Priority, Preferences, Type, Time, and Remarks. 

Morning Exercise is priority 1, which is the highest. It's scheduled at 6 AM but the remark says an urgent task came up. That probably means they missed it. Since it's high priority, maybe they need to adjust their schedule to accommodate both the exercise and the new task.

Read Books is priority 2, lower than exercise. It's at 8 PM. They didn't provide any remarks, so maybe they're consistent, but I should chec

In [10]:
ans = Retriever()

In [11]:
ans

[(1,
  'Morning Exercise',
  1,
  5,
  'Health',
  '06:00',
  'An urgent new task has come up.'),
 (2, 'Read Books', 2, 4, 'Learning', '20:00', None),
 (3, 'Practice Piano', 3, 3, 'Creativity', '15:00', None)]